# GAN on Cat Faces (64x64)

DCGAN trained on the [Cats Faces 64x64 for Generative Models](https://www.kaggle.com/datasets/spandan2/cats-faces-64x64-for-generative-models) Kaggle dataset.

This version adds:
- Fixed noise vector so the sample grid tracks the *same* latent points across epochs
- Checkpointing every N epochs + resume-from-checkpoint support
- Loss curve tracking and plotting
- A standalone inference cell to generate images from any saved checkpoint

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt

## 0. Downloading Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("spandan2/cats-faces-64x64-for-generative-models")  # Download latest version
print("Path to dataset files:", path)

## 1. Hyperparameters

In [ ]:
latent_dim = 100
image_size = 64
channels = 3
the_batch_size = 64
num_epochs = 10
lr = 0.0002
beta1 = 0.5

# New: checkpointing / output config
checkpoint_dir = "checkpoints"
sample_dir = "samples"
save_every = 2          # save a checkpoint every N epochs
resume_training = False  # set True to continue from the latest checkpoint
num_fixed_samples = 16   # how many images to track across epochs

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(sample_dir, exist_ok=True)

## 2. Generator Network

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is latent_dim
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            # State size: 512 x 4 x 4
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # State size: 256 x 8 x 8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # State size: 128 x 16 x 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # State size: 64 x 32 x 32
            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False),
            nn.Tanh()
            # Output size: 3 x 64 x 64
        )

    def forward(self, x):
        return self.main(x)

## 3. Discriminator Network

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # Input size: 3 x 64 x 64
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # State size: 64 x 32 x 32
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # State size: 128 x 16 x 16
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # State size: 256 x 8 x 8
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # State size: 512 x 4 x 4
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x)

## 4. Weight Initialization

Standard DCGAN init (mean=0, std=0.02 on conv/batchnorm layers). Not required, but usually gives more stable training than PyTorch's default init.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 5. Checkpoint Helpers

Checkpoints store both model states, both optimizer states, the epoch number, and the loss history, so resuming continues training exactly rather than just reloading weights.

In [ ]:
def save_checkpoint(path, epoch, generator, discriminator, g_optimizer, d_optimizer, g_losses, d_losses, fixed_noise):
    torch.save({
        "epoch": epoch,
        "generator_state": generator.state_dict(),
        "discriminator_state": discriminator.state_dict(),
        "g_optimizer_state": g_optimizer.state_dict(),
        "d_optimizer_state": d_optimizer.state_dict(),
        "g_losses": g_losses,
        "d_losses": d_losses,
        "fixed_noise": fixed_noise,
    }, path)
    print(f"Saved checkpoint: {path}")


def load_checkpoint(path, generator, discriminator, g_optimizer, d_optimizer, device):
    ckpt = torch.load(path, map_location=device)
    generator.load_state_dict(ckpt["generator_state"])
    discriminator.load_state_dict(ckpt["discriminator_state"])
    g_optimizer.load_state_dict(ckpt["g_optimizer_state"])
    d_optimizer.load_state_dict(ckpt["d_optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    g_losses = ckpt.get("g_losses", [])
    d_losses = ckpt.get("d_losses", [])
    fixed_noise = ckpt.get("fixed_noise", None)
    print(f"Resumed from {path} (starting at epoch {start_epoch})")
    return start_epoch, g_losses, d_losses, fixed_noise


def latest_checkpoint(checkpoint_dir):
    if not os.path.isdir(checkpoint_dir):
        return None
    ckpts = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_epoch_") and f.endswith(".pth")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda f: int(f.split("_")[-1].split(".")[0]))
    return os.path.join(checkpoint_dir, ckpts[-1])

## 6. Training Function

In [ ]:
def train_gan(resume=False):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Data preprocessing
    transform = T.Compose([
        T.Resize(image_size),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    # Load dataset
    dataset = ImageFolder(root="/root/.cache/kagglehub/datasets/spandan2/cats-faces-64x64-for-generative-models/versions/1", transform=transform)
    dataloader = DataLoader(dataset, batch_size=the_batch_size, shuffle=True, num_workers=2)

    # Initialize networks and optimizers
    generator = Generator().to(device)
    discriminator = Discriminator().to(device)
    generator.apply(weights_init)
    discriminator.apply(weights_init)

    criterion = nn.BCELoss()
    g_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

    # Fixed noise: sampled once, reused every epoch so the sample grid tracks
    # the same latent points across training instead of unrelated random draws
    fixed_noise = torch.randn(num_fixed_samples, latent_dim, 1, 1, device=device)

    g_losses, d_losses = [], []
    start_epoch = 0

    if resume:
        ckpt_path = latest_checkpoint(checkpoint_dir)
        if ckpt_path is not None:
            start_epoch, g_losses, d_losses, loaded_noise = load_checkpoint(
                ckpt_path, generator, discriminator, g_optimizer, d_optimizer, device
            )
            if loaded_noise is not None:
                fixed_noise = loaded_noise.to(device)
        else:
            print("No checkpoint found, starting from scratch.")

    # Training loop
    for epoch in range(start_epoch, num_epochs):
        for i, (real_images, _) in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Train Discriminator
            d_optimizer.zero_grad()
            label_real = torch.ones(batch_size, 1).to(device)
            label_fake = torch.zeros(batch_size, 1).to(device)

            output_real = discriminator(real_images).view(-1, 1)
            d_loss_real = criterion(output_real, label_real)

            noise = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_images = generator(noise)
            output_fake = discriminator(fake_images.detach()).view(-1, 1)
            d_loss_fake = criterion(output_fake, label_fake)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            d_optimizer.step()

            # Train Generator
            g_optimizer.zero_grad()
            output_fake = discriminator(fake_images).view(-1, 1)
            g_loss = criterion(output_fake, label_real)
            g_loss.backward()
            g_optimizer.step()

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

            if i % 100 == 0:
                print(f'Epoch [{epoch}/{num_epochs}] Batch [{i}/{len(dataloader)}] '
                      f'd_loss: {d_loss.item():.4f} g_loss: {g_loss.item():.4f}')

        # Generate from the fixed noise vector so samples are comparable epoch-to-epoch
        with torch.no_grad():
            fake_samples = generator(fixed_noise).cpu()
            sample_path = os.path.join(sample_dir, f'fake_cats_epoch_{epoch}.png')
            torchvision.utils.save_image(fake_samples, sample_path, normalize=True, nrow=4)

            grid_img = T.ToPILImage()(torchvision.utils.make_grid(fake_samples, normalize=True, nrow=4))
            plt.imshow(grid_img)
            plt.axis("off")
            plt.title(f"Generated Images at Epoch {epoch}")
            plt.show()
            print(f'Training Epoch [{epoch}] finished.\n')

        # Periodic checkpoint
        if (epoch + 1) % save_every == 0 or epoch == num_epochs - 1:
            ckpt_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}.pth')
            save_checkpoint(ckpt_path, epoch, generator, discriminator,
                             g_optimizer, d_optimizer, g_losses, d_losses, fixed_noise)

    # Always save final plain weights too, for easy loading in inference
    torch.save(generator.state_dict(), 'generator.pth')
    torch.save(discriminator.state_dict(), 'discriminator.pth')
    print("Saved final generator.pth / discriminator.pth")

    return generator, discriminator, g_losses, d_losses

## 7. Loss Curve Plotting

In [ ]:
def plot_losses(g_losses, d_losses, save_path="loss_curve.png"):
    plt.figure(figsize=(10, 5))
    plt.plot(g_losses, label="Generator")
    plt.plot(d_losses, label="Discriminator")
    plt.xlabel("Training iteration")
    plt.ylabel("Loss")
    plt.title("GAN Training Losses")
    plt.legend()
    plt.savefig(save_path)
    plt.show()
    print(f"Saved loss curve to {save_path}")

## 8. Standalone Inference

Generate new images from any saved checkpoint (either a full training checkpoint or a plain `generator.pth` weights file), without needing to re-run training.

In [ ]:
def generate_images(checkpoint_path='generator.pth', num_images=16, output_path='generated_cats.png'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator = Generator().to(device)

    state = torch.load(checkpoint_path, map_location=device)
    # Support both a full training checkpoint dict and a plain state_dict file
    if isinstance(state, dict) and "generator_state" in state:
        generator.load_state_dict(state["generator_state"])
    else:
        generator.load_state_dict(state)

    generator.eval()

    with torch.no_grad():
        noise = torch.randn(num_images, latent_dim, 1, 1, device=device)
        fake_images = generator(noise).cpu()
        torchvision.utils.save_image(fake_images, output_path, normalize=True, nrow=4)

        grid_img = T.ToPILImage()(torchvision.utils.make_grid(fake_images, normalize=True, nrow=4))
        plt.imshow(grid_img)
        plt.axis("off")
        plt.title("Generated Images")
        plt.show()
        print(f"Saved generated images to {output_path}")

## 9. Run Training

In [ ]:
print("Training GAN model...")
generator, discriminator, g_losses, d_losses = train_gan(resume=resume_training)

## 10. Plot Loss Curves

In [ ]:
plot_losses(g_losses, d_losses)

## 11. Generate Images From a Saved Checkpoint

Run this independently of training (e.g. in a fresh session) to produce new samples from a checkpoint on disk.

In [ ]:
# Example: generate from the final plain weights file saved after training
generate_images(checkpoint_path='generator.pth', num_images=16, output_path='generated_cats.png')

# Example: generate from a specific epoch checkpoint instead
# generate_images(checkpoint_path='checkpoints/checkpoint_epoch_9.pth', num_images=16, output_path='generated_cats_epoch9.png')

## 12. Resume Training (Train More Epochs)

Use this to continue training an already-trained model instead of starting over -- e.g. you trained 10 epochs earlier and now want a total of 20.

**How it works:** set `num_epochs` to the new *total* you want to reach (not "how many more"), set `resume_training = True`, then call `train_gan(resume=True)` again. It finds the latest file in `checkpoints/`, restores the generator, discriminator, both optimizer states, the loss history, and the fixed noise vector, and continues the loop from `checkpoint_epoch + 1` up to `num_epochs`.

**Gotcha:** if `num_epochs` is left at the old total, `start_epoch` will already equal it, so the loop runs zero times. Always raise `num_epochs` to the new target before running this cell.

In [ ]:
# Example: previously trained to epoch 10, now train up to epoch 20 total.
# Edit these two values for your situation, then run this cell.

num_epochs = 20          # <-- set this to the NEW TOTAL number of epochs, not "how many more"
resume_training = True   # <-- must be True to continue from the last checkpoint

print(f"Resuming training up to epoch {num_epochs}...")
generator, discriminator, g_losses, d_losses = train_gan(resume=resume_training)

plot_losses(g_losses, d_losses)